# 01 - Observation Pipeline Debug
Verify obs vector is correctly packed, normalized, interpretable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pufferlib.ocean.drive.drive import Drive
from pufferlib.ocean.drive import binding
import pufferlib.viz

# --- Environment configuration ---
NUM_AGENTS = 64
SIMULATION_MODE = "gigaflow"
DYNAMICS_MODEL = "jerk"
ACTION_TYPE = "discrete"
DT = 0.1
SCENARIO_LENGTH = 512
RESAMPLE_FREQUENCY = 0
REWARD_CONDITIONING = True
REWARD_RANDOMIZATION = False
TARGET_TYPE = "static"
COLLISION_BEHAVIOR = 1
OFFROAD_BEHAVIOR = 1
SEED = 42
MAP_DIR = "../pufferlib/resources/drive/binaries/carla"

# --- Observation dimensions (configurable) ---
MAX_PARTNERS = 16
MAX_LANES = 32
MAX_BOUNDS = 32
MAX_TRAFFIC = 4

# --- Derived from binding (compile-time) ---
EGO_DIM = binding.EGO_FEATURES_JERK
NUM_COEFS = binding.NUM_REWARD_COEFS
PARTNER_F = binding.PARTNER_FEATURES
ROAD_F = binding.ROAD_FEATURES
TRAFFIC_CONTROL_F = binding.TRAFFIC_CONTROL_FEATURES
NUM_TRAFFIC_CONTROL_TYPES = binding.NUM_TRAFFIC_CONTROL_TYPES
COEF_NAMES = [
    "goal_radius",
    "collision",
    "offroad",
    "comfort",
    "lane_align",
    "lane_center",
    "velocity",
    "traffic_light",
    "center_bias",
    "vel_align",
    "overspeed",
    "timestep",
    "reverse",
    "throttle",
    "steer",
    "acc",
]

# --- Create environment ---
env = Drive(
    num_agents=NUM_AGENTS,
    num_maps=1,
    min_agents_per_env=NUM_AGENTS,
    max_agents_per_env=NUM_AGENTS,
    simulation_mode=SIMULATION_MODE,
    dynamics_model=DYNAMICS_MODEL,
    action_type=ACTION_TYPE,
    dt=DT,
    scenario_length=SCENARIO_LENGTH,
    resample_frequency=RESAMPLE_FREQUENCY,
    reward_conditioning=REWARD_CONDITIONING,
    reward_randomization=REWARD_RANDOMIZATION,
    target_type=TARGET_TYPE,
    map_dir=MAP_DIR,
    collision_behavior=COLLISION_BEHAVIOR,
    offroad_behavior=OFFROAD_BEHAVIOR,
    obs_slots_lane=MAX_LANES,
    obs_slots_boundary=MAX_BOUNDS,
    obs_slots_partners=MAX_PARTNERS,
    obs_slots_traffic_controls=MAX_TRAFFIC,
    seed=SEED,
)
obs, info = env.reset(seed=SEED)

# --- Derived from env ---
MAX_TARGET = env.num_target_waypoints
TARGET_F = binding.STATIC_TARGET_FEATURES if TARGET_TYPE == "static" else binding.DYNAMIC_TARGET_FEATURES
TARGET_DIM = MAX_TARGET * TARGET_F

print(f"obs shape: {obs.shape}, dtype: {obs.dtype}")
print(f"EGO_DIM={EGO_DIM}, NUM_COEFS={NUM_COEFS}, MAX_PARTNERS={MAX_PARTNERS}, PARTNER_F={PARTNER_F}")
print(f"MAX_LANES={MAX_LANES}, MAX_BOUNDS={MAX_BOUNDS}, ROAD_F={ROAD_F}")
print(f"MAX_TRAFFIC={MAX_TRAFFIC}, TRAFFIC_F={TRAFFIC_CONTROL_F}")

## Raw obs inspection

In [ ]:
# Take first step so obs are populated
actions = np.zeros([env.num_agents, 1], dtype=np.int64)

obs, rew, term, trunc, info = env.step(actions)

print(f"shape: {obs.shape}, dtype: {obs.dtype}")
print(f"min: {obs.min():.4f}, max: {obs.max():.4f}, mean: {obs.mean():.4f}, std: {obs.std():.4f}")
print(f"NaN: {np.isnan(obs).sum()}, Inf: {np.isinf(obs).sum()}")
print(f"% zeros: {(obs == 0).mean() * 100:.1f}%")
print(f"% outside [-1,1]: {((obs < -1) | (obs > 1)).mean() * 100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(obs.flatten(), bins=100, edgecolor="black", alpha=0.7)
axes[0].set_title("Full obs distribution")
axes[0].set_xlabel("Value")
# Per-agent: show obs[0] vs obs[1]
for i in range(min(4, obs.shape[0])):
    axes[1].plot(obs[i], alpha=0.5, label=f"agent {i}")
axes[1].set_title("Obs vector by index (first 4 agents)")
axes[1].legend()
plt.tight_layout()
plt.show()

## Unpack with pufferlib.viz.unpack_obs

In [ ]:
ego, target, partners, lanes, boundaries, traffic = pufferlib.viz.unpack_obs(
    obs[:1],
    dynamics_model=DYNAMICS_MODEL,
    target_type=TARGET_TYPE,
    reward_conditioning=REWARD_CONDITIONING,
    num_target_waypoints=env.num_target_waypoints,
    max_partners=MAX_PARTNERS,
    max_lane_segments=MAX_LANES,
    max_boundary_segments=MAX_BOUNDS,
    obs_slots_traffic_controls=MAX_TRAFFIC,
)
print(f"ego: {ego.shape} = {ego}")
print(f"target: {target.shape}")
print(f"partners: {partners.shape}")
print(f"lanes: {lanes.shape}")
print(f"boundaries: {boundaries.shape}")
print(f"traffic: {traffic.shape}")


labels = [
    "speed",
    "width",
    "length",
    "steering",
    "a_long",
    "a_lat",
    "lane_center_dist_01",
    "lane_heading_cos",
    "speed_limit",
]
for name, val in zip(labels, ego):
    print(f"  {name}: {val:.4f}")

## Manual slice verification

In [ ]:
o = obs[0]  # first agent flat obs
idx = 0

# Ego
ego_manual = o[idx : idx + EGO_DIM]
idx += EGO_DIM
assert np.allclose(ego_manual, ego), f"ego mismatch: {ego_manual} vs {ego}"

# Reward conditioning coefs
coefs_manual = o[idx : idx + NUM_COEFS]
idx += NUM_COEFS

# Target
target_manual = o[idx : idx + MAX_TARGET * TARGET_F].reshape(MAX_TARGET, TARGET_F)
idx += MAX_TARGET * TARGET_F
assert np.allclose(target_manual, target), "target mismatch"

# Partners
partners_manual = o[idx : idx + MAX_PARTNERS * PARTNER_F].reshape(MAX_PARTNERS, PARTNER_F)
idx += MAX_PARTNERS * PARTNER_F
assert np.allclose(partners_manual, partners), "partners mismatch"

# Lanes
lanes_manual = o[idx : idx + MAX_LANES * ROAD_F].reshape(MAX_LANES, ROAD_F)
idx += MAX_LANES * ROAD_F
assert np.allclose(lanes_manual, lanes), "lanes mismatch"

# Boundaries
bounds_manual = o[idx : idx + MAX_BOUNDS * ROAD_F].reshape(MAX_BOUNDS, ROAD_F)
idx += MAX_BOUNDS * ROAD_F
assert np.allclose(bounds_manual, boundaries), "boundaries mismatch"

# Traffic
traffic_manual = o[idx : idx + MAX_TRAFFIC * TRAFFIC_CONTROL_F].reshape(MAX_TRAFFIC, TRAFFIC_CONTROL_F)
idx += MAX_TRAFFIC * TRAFFIC_CONTROL_F
assert np.allclose(traffic_manual, traffic), "traffic mismatch"

assert idx == obs.shape[1], f"obs size mismatch: used {idx}, total {obs.shape[1]}"
print(f"All slices match. Total features used: {idx}")

## Reward conditioning coefficients

In [ ]:
coefs = obs[0, EGO_DIM : EGO_DIM + NUM_COEFS]
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(NUM_COEFS), coefs, tick_label=COEF_NAMES)
ax.set_ylabel("Normalized coef value")
ax.set_title("Reward conditioning coefficients (agent 0)")
plt.xticks(rotation=45, ha="right")
for bar, val in zip(bars, coefs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

# Compare across agents
all_coefs = obs[:, EGO_DIM : EGO_DIM + NUM_COEFS]
print("Coef stats across agents:")
for i, name in enumerate(COEF_NAMES):
    c = all_coefs[:, i]
    print(f"  {name:15s}: mean={c.mean():.3f} std={c.std():.3f} min={c.min():.3f} max={c.max():.3f}")

## Partner observations

In [ ]:
partner_labels = ["rel_x", "rel_y", "rel_z", "length", "width", "heading_cos", "heading_sin", "speed"]
active_mask = ~np.all(partners == 0, axis=1)
n_active = active_mask.sum()
print(f"Active partners: {n_active}/{MAX_PARTNERS}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = axes[0].imshow(partners, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_xticks(range(PARTNER_F))
axes[0].set_xticklabels(partner_labels, rotation=45, ha="right")
axes[0].set_ylabel("Partner index")
axes[0].set_title(f"Partner obs heatmap ({n_active} active)")
plt.colorbar(im, ax=axes[0])

# Scatter in ego frame
active_partners = partners[active_mask]
if len(active_partners) > 0:
    axes[1].scatter(active_partners[:, 0], active_partners[:, 1], c="gray", s=100, edgecolors="black")
    for i, p in enumerate(active_partners):
        axes[1].annotate(str(i), (p[0], p[1]), fontsize=8, ha="center", va="bottom")
axes[1].scatter(0, 0, c="blue", s=200, marker="s", label="ego", zorder=10)
axes[1].set_xlabel("rel_x")
axes[1].set_ylabel("rel_y")
axes[1].set_title("Partners in ego frame")
axes[1].legend()
axes[1].set_aspect("equal")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Lane / boundary segments

In [ ]:
road_labels = ["rel_x", "rel_y", "rel_z", "length", "width", "dir_cos", "dir_sin"]

lane_active = ~np.all(lanes == 0, axis=1)
bound_active = ~np.all(boundaries == 0, axis=1)
print(f"Active lanes: {lane_active.sum()}/{MAX_LANES}, boundaries: {bound_active.sum()}/{MAX_BOUNDS}")

fig, ax = plt.subplots(figsize=(10, 10))

# Mirror the canonical road rendering in pufferlib.viz.plot_observation
for seg in lanes[lane_active]:
    x, y, z, length, width, dc, ds = seg
    ax.scatter(x, y, color="lightgrey", s=10, zorder=1)
    ax.plot(
        [x + dc * length / 2, x - dc * length / 2],
        [y + ds * length / 2, y - ds * length / 2],
        color="lightgrey",
        linewidth=1,
        zorder=1,
    )

for seg in boundaries[bound_active]:
    x, y, z, length, width, dc, ds = seg
    ax.scatter(x, y, color="black", s=10, zorder=1)
    ax.plot(
        [x + dc * length / 2, x - dc * length / 2],
        [y + ds * length / 2, y - ds * length / 2],
        color="black",
        linewidth=1,
        zorder=1,
    )

ax.scatter(0, 0, color="blue", s=200, marker="s", label="ego", zorder=10)
ax.text(
    0.12,
    0.95,
    f"Lanes: {lane_active.sum()}\nBoundaries: {bound_active.sum()}",
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
)
ax.axis((-1, 1, -1, 1))
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("X (ego frame)")
ax.set_ylabel("Y (ego frame)")
ax.set_title("Lane + boundary segments in ego frame")
plt.tight_layout()
plt.show()

## Ego-centric view (pufferlib.viz)

In [ ]:
img = pufferlib.viz.plot_observation(
    obs[:1],
    dynamics_model=DYNAMICS_MODEL,
    target_type=TARGET_TYPE,
    reward_conditioning=True,
    num_target_waypoints=env.num_target_waypoints,
    max_partners=MAX_PARTNERS,
    max_lane_segments=MAX_LANES,
    max_boundary_segments=MAX_BOUNDS,
    obs_slots_traffic_controls=MAX_TRAFFIC,
)
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(img)
ax.axis("off")
ax.set_title("Ego-centric observation (agent 0)")
plt.tight_layout()
plt.show()

## Bird's eye view (simulator state)

In [ ]:
scenarios = env.get_state()
# get_state returns a list of scenario dicts (one per sub-env) or a single dict
if isinstance(scenarios, list):
    scenario = scenarios[0]
else:
    scenario = scenarios

img = pufferlib.viz.plot_simulator_state(scenario, timestep=0)
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(img)
ax.axis("off")
ax.set_title("Bird's eye view")
plt.tight_layout()
plt.show()

## Multi-step: ego features over time

In [ ]:
N_STEPS = 20
ego_labels = [
    "speed",
    "width",
    "length",
    "steering",
    "a_long",
    "a_lat",
    "lane_center_dist",
    "lane_heading_cos",
    "speed_limit",
]
ego_history = np.zeros((N_STEPS, EGO_DIM))

for t in range(N_STEPS):
    actions = np.zeros([env.num_agents, 1], dtype=np.int64)
    obs_t, _, _, _, _ = env.step(actions)
    ego_history[t] = obs_t[0, :EGO_DIM]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# Speed
axes[0, 0].plot(ego_history[:, 0])
axes[0, 0].set_title("speed")
axes[0, 0].set_xlabel("step")
# Steering
axes[0, 1].plot(ego_history[:, 3])
axes[0, 1].set_title("steering")
axes[0, 1].set_xlabel("step")
# a_long
axes[1, 0].plot(ego_history[:, 4])
axes[1, 0].set_title("a_long")
axes[1, 0].set_xlabel("step")
# a_lat
axes[1, 1].plot(ego_history[:, 5])
axes[1, 1].set_title("a_lat")
axes[1, 1].set_xlabel("step")
plt.suptitle("Agent 0 ego features over 20 steps (no-op action)")
plt.tight_layout()
plt.show()

## Cross-agent distributions

In [ ]:
# Current obs across all agents
# Ego features (jerk): speed(0), width(1), length(2), steering(3), a_long(4), a_lat(5), lane_center(6), lane_heading(7), speed_limit(8)
speeds = obs[:, 0]  # speed is at index 0

# Target waypoints start after ego + reward coefs
target_start = EGO_DIM + NUM_COEFS
# Each target waypoint has TARGET_F features; first two are rel_x, rel_y
first_target_x = obs[:, target_start]
first_target_y = obs[:, target_start + 1]
target_dists = np.sqrt(first_target_x**2 + first_target_y**2)

# Count active partners per agent
partner_start = EGO_DIM + NUM_COEFS + TARGET_DIM
partner_end = partner_start + MAX_PARTNERS * PARTNER_F
all_partners = obs[:, partner_start:partner_end].reshape(-1, MAX_PARTNERS, PARTNER_F)
partner_counts = (~np.all(all_partners == 0, axis=2)).sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(speeds, bins=20, edgecolor="black", alpha=0.7)
axes[0].set_title(f"Speed distribution (N={len(speeds)})")
axes[0].set_xlabel("speed")

axes[1].hist(target_dists, bins=20, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_title("Distance to first target waypoint")
axes[1].set_xlabel("distance")

axes[2].hist(partner_counts, bins=range(MAX_PARTNERS + 2), edgecolor="black", alpha=0.7, color="green")
axes[2].set_title("Active partners per agent")
axes[2].set_xlabel("count")
plt.tight_layout()
plt.show()